In [1]:
import sys
sys.path.append("/kaggle/input/datasets/yusufhilal/full-model/Event-Extraction-Model")

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
import numpy as np
import pandas as pd
import torch
import random

from sklearn.model_selection import GroupShuffleSplit, KFold, GroupKFold
from transformers import AutoTokenizer

from helpers.utils import load_config, compute_num_stats
from helpers.dataset import merge_labels
from helpers.losses import bio_loss, weighted_bio_loss
from helpers.metrics import find_best_threshold_peak, boundary_metrics_peak, collect_page_probs_and_truth, pick_starts_from_probs, start_prf_with_tolerance, compute_prf
from helpers.train_utils import make_loader, run_epoch

from models.dom_extractor import init_model_and_optim, set_bert_trainable

In [4]:
# config --------------------------------------------------------------
cfg = load_config()

cfg["model"]["d_model"] = 64 # can test 96 later
cfg["model"]["nhead"] = 4
cfg["model"]["num_layers"] = 4
cfg["model"]["dropout"] = 0.3
cfg["model"]["max_tokens"] = 32 # drop to 32 if facing memory issues
cfg["model"]["use_tag"] = False
# cfg["model"]["use_parent_tag"] = False

cfg["training"]["epochs"] = 30
cfg["training"]["freeze_epochs"] = 8
cfg["training"]["batch_size"] = 1 # fixed
# cfg["training"]["lr_bert"] = 5e-6
# cfg["training"]["lr_other"] = 1e-4
# cfg["training"]["weight_decay"] = 0.01
cfg["training"]["n_splits"] = 5 # 1 for testing, 3 for improvements, 5 for final
cfg["training"]["seed"] = 42

cfg["inference"]["nms_k"] = 1
cfg["inference"]["min_gap"] = 2
cfg["inference"]["tol"] = 1

# cfg["features"]["num_cols"] = []
# cfg["features"]["num_cols"] = ["depth", "sibling_index", "children_count", "same_tag_sibling_count", "same_text_sibling_count"] # structural num
# cfg["features"]["bool_cols"] = ["has_link", "link_is_absolute", "parent_has_link", "is_leaf", "has_class", "has_id",  # removing date/time signals
                               #  "attr_has_word_name", "attr_has_word_date", "attr_has_word_time", "attr_has_word_location", 
                               #  "attr_has_word_link", "text_has_word_name", "text_has_word_date", "text_word_time", 
                               #  "text_word_description", "text_word_location"
                               # ]
# cfg["features"]["bool_cols"] = ["has_link", "link_is_absolute", "parent_has_link", "is_leaf", "has_class", "has_id"] # structural bools

In [5]:
SEED = cfg["training"]["seed"]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda:0")
print("Device:", DEVICE)

Device: cuda:0


In [6]:
# load data and preprocess -----------------------------------------------
df = pd.read_csv("/kaggle/input/datasets/yusufhilal/full-model/Event-Extraction-Model/data/full_data.csv")

df = merge_labels(df, "label")
df = df.sort_values(["source", "rendering_order"]).reset_index(drop=True) # extra cleaninig

df["in_event"] = df["event_id"].notna().astype(int) # binary event flag column
df["start_event"] = 0

m = df["event_id"].notna() # mask for events only

# take index of first row of each event (event boundary)
first_idx = df.loc[m].groupby(["source", "event_id"], sort=False).head(1).index 
df.loc[first_idx, "start_event"] = 1
# df["start_event"] = df["start_event"].fillna(0).astype(int)

# outside, begin, inside tagging
df["bio"] = 0 # outside
df.loc[df["in_event"].eq(1), "bio"] = 2 # events inside
df.loc[df["start_event"].eq(1), "bio"] = 1 # boundary -> beginning

print(f"Pages: {df['source'].nunique()}")
print(f"Total nodes: {len(df)}")
print(f"Label counts:\n{df['label'].value_counts()}")

Pages: 15
Total nodes: 3100
Label counts:
label
Other          2438
Date            242
Location        139
Name            120
Time            116
Description      27
Price            18
Name: count, dtype: int64


In [7]:
# Vocab and Feature Setup ---------------------------------------------------------------
# sort labels, make them integers for training and return mapping to labels for eval
LABELS = sorted(df["label"].unique().tolist())
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

TAG_VOCAB = {t: i for i, t in enumerate(sorted(df["tag"].astype(str).unique().tolist()))}
PARENT_TAG_VOCAB = {t: i for i, t in enumerate(sorted(df["parent_tag"].astype(str).unique().tolist()))}

NUM_COLS = cfg["features"]["num_cols"]
BOOL_COLS = cfg["features"]["bool_cols"]

print(f"Labels: {LABELS}")
print(f"Tag vocab size: {len(TAG_VOCAB)}")
print(f"Parent tag vocab size: {len(PARENT_TAG_VOCAB)}")
print(f"Num features: {len(NUM_COLS)}")
print(f"Bool features: {len(BOOL_COLS)}")

Labels: ['Date', 'Description', 'Location', 'Name', 'Other', 'Price', 'Time']
Tag vocab size: 15
Parent tag vocab size: 19
Num features: 11
Bool features: 20


In [8]:
def augment_by_event_mixing(df, n_synthetic=10, seed=42):
    rng = np.random.default_rng(seed)
    
    # collect all individual events across all pages
    events = []
    for source, page_df in df.groupby("source"):
        page_df = page_df.sort_values("rendering_order").reset_index(drop=True)
        event_ids = page_df[page_df["event_id"].notna()]["event_id"].unique()
        for eid in event_ids:
            event_nodes = page_df[page_df["event_id"] == eid].copy()
            events.append(event_nodes)
    
    synthetic_pages = []
    for i in range(n_synthetic):
        # pick 3-8 random events
        n_events = rng.integers(3, 9)
        chosen = [events[j].copy() for j in rng.choice(len(events), n_events, replace=False)]
        
        # stitch together with correct bio labels
        page_nodes = []
        for k, event_df in enumerate(chosen):
            event_df = event_df.copy()
            event_df["bio"] = 2  # all I
            event_df.iloc[0, event_df.columns.get_loc("bio")] = 1  # first node is B
            page_nodes.append(event_df)
        
        synthetic_page = pd.concat(page_nodes).reset_index(drop=True)
        synthetic_page["source"] = f"synthetic_{i}"
        synthetic_page["rendering_order"] = range(len(synthetic_page))
        synthetic_pages.append(synthetic_page)
    
    return pd.concat([df] + synthetic_pages).reset_index(drop=True)

In [9]:
# Train/Test Split ------------------------------------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.11, random_state=cfg["training"]["seed"])
train_idx, test_idx = next(gss.split(df, groups=df["source"]))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

train_df = augment_by_event_mixing(train_df, n_synthetic=20, seed=SEED)

cv_sources = train_df["source"].unique()
test_sources = test_df["source"].unique()

print(f"\nTrain pages: {len(cv_sources)}")
print("\nHoldout TEST pages:", len(test_sources), sorted(list(test_sources)))


Train pages: 33

Holdout TEST pages: 2 ['neacac_spring.net_pattern_labeled', 'pnacac_spring.org_pattern_labeled']


In [10]:
# Tokenizer -------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(cfg["model"]["name"])

In [11]:
# # Cross Validation -----------------------------------------------------
# N_SPLITS = min(cfg["training"]["n_splits"], len(cv_sources))
# cv_results = []

# if cfg["training"]["n_splits"] == 1:
#     # single fold - just do a quick train/val split
#     gss_cv = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=cfg["training"]["seed"])
#     splits = list(gss_cv.split(train_df, groups=train_df["source"]))
#     kf_splits = [(splits[0][0], splits[0][1])]
# else:
#     kf = GroupKFold(n_splits=N_SPLITS)
#     kf_splits = list(kf.split(train_df, groups=train_df["source"]))


# for fold, (tr_idx, va_idx) in enumerate(kf_splits, start=1):
        
#     torch.cuda.empty_cache()

#     fold_train_df = train_df.iloc[tr_idx].copy()
#     fold_val_df = train_df.iloc[va_idx].copy()

#     # fold_train_df = augment_by_event_mixing(fold_train_df, n_synthetic=20, seed=SEED+fold)
    
#     num_mean, num_std = compute_num_stats(fold_train_df, NUM_COLS)
#     loss_fn = bio_loss(DEVICE)
#     # loss_fn = weighted_bio_loss(fold_train_df, DEVICE)
#     # loss_fn = weighted_bio_loss(fold_train_df, DEVICE, weight_cap=5.0)
    
#     print(f"\n===== Fold {fold}/{N_SPLITS} =====")
#     print(f"Train pages: {fold_train_df['source'].nunique()} Val pages: {fold_val_df['source'].nunique()}")

#     train_loader = make_loader(
#         fold_train_df, tokenizer,
#         TAG_VOCAB, PARENT_TAG_VOCAB,
#         NUM_COLS, BOOL_COLS,
#         num_mean, num_std,
#         batch_size=cfg["training"]["batch_size"],
#         max_tokens=cfg["model"]["max_tokens"],
#         shuffle=True
#     )
    
#     val_loader = make_loader(
#         fold_val_df, tokenizer,
#         TAG_VOCAB, PARENT_TAG_VOCAB,
#         NUM_COLS, BOOL_COLS,
#         num_mean, num_std,
#         batch_size=cfg["training"]["batch_size"],
#         max_tokens=cfg["model"]["max_tokens"],
#         shuffle=False
#     )

#     model, optimizer = init_model_and_optim(
#         cfg,
#         tag_vocab_size=len(TAG_VOCAB),
#         parent_tag_vocab_size=len(PARENT_TAG_VOCAB),
#         num_numeric_features=len(NUM_COLS),
#         num_bool_features=len(BOOL_COLS),
#         device=DEVICE
#     )
    
#     set_bert_trainable(model, False)
#     best = {"f1": -1.0, "th": 0.5, "state": None}

#     for epoch in range(cfg["training"]["epochs"]):

#         if epoch == cfg["training"]["freeze_epochs"]:
#             set_bert_trainable(model, True)

#         tr_loss = run_epoch(model, optimizer, train_loader, loss_fn, DEVICE, training=True)
#         val_loss = run_epoch(model, optimizer, val_loader, loss_fn, DEVICE, training=False)

#         th, f1 = find_best_threshold_peak(val_loader, model, DEVICE,
#                                           nms_k=cfg["inference"]["nms_k"],
#                                           min_gap=cfg["inference"]["min_gap"],
#                                           tol=cfg["inference"]["tol"]
#                                           )

#         if f1 > best["f1"]:
#             best["f1"] = f1
#             best["th"] = th
#             best["state"] ={k: v.cpu() for k, v in model.state_dict().items()}
            
#         if (epoch + 1) % 5 == 0:
#             print(f"Epoch {epoch+1:02d} | tr_loss={tr_loss:.4f} val_loss={val_loss:.4f} best_f1={best['f1']:.4f} best_th={best['th']:.4f}")


#     model.load_state_dict(best["state"])

#     bp, br, bf1 = boundary_metrics_peak(
#         val_loader, model, DEVICE,
#         threshold=best["th"],
#         nms_k=cfg["inference"]["nms_k"],
#         min_gap=cfg["inference"]["min_gap"],
#         tol=cfg["inference"]["tol"]
#     )
    
#     print(f"Fold {fold} | P={bp:.4f} R={br:.4f} F1={bf1:.4f} (th={best['th']:.4f})")

#     cv_results.append({
#         "bf1": bf1,
#         "th": best["th"],
#     })

#     del model, optimizer
#     torch.cuda.empty_cache()

# # CV summary
# mean_bf1 = float(np.mean([x["bf1"] for x in cv_results]))
# mean_th = float(np.mean([x["th"] for x in cv_results]))
# best_th_cv = mean_th

# print(f"\n===== CV Summary =====")
# print(f"Mean F1: {mean_bf1:.4f}")
# print(f"Mean threshold: {mean_th:.4f}")
# print(f"Using threshold for final eval: {best_th_cv:.4f}")


===== Fold 1/5 =====
Train pages: 30 Val pages: 3


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 05 | tr_loss=0.3748 val_loss=0.2989 best_f1=0.7636 best_th=0.0150
Epoch 10 | tr_loss=0.1970 val_loss=0.0710 best_f1=0.8627 best_th=0.0750
Epoch 15 | tr_loss=0.0890 val_loss=0.0689 best_f1=0.8929 best_th=0.0210
Epoch 20 | tr_loss=0.0441 val_loss=0.0204 best_f1=0.9434 best_th=0.0150
Epoch 25 | tr_loss=0.0232 val_loss=0.0245 best_f1=0.9434 best_th=0.0150
Epoch 30 | tr_loss=0.0251 val_loss=0.0247 best_f1=0.9434 best_th=0.0150
Fold 1 | P=1.0000 R=0.8929 F1=0.9434 (th=0.0150)

===== Fold 2/5 =====
Train pages: 27 Val pages: 6


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 05 | tr_loss=0.3754 val_loss=0.3903 best_f1=0.6800 best_th=0.0490
Epoch 10 | tr_loss=0.1760 val_loss=0.2150 best_f1=0.7955 best_th=0.1090
Epoch 15 | tr_loss=0.0737 val_loss=0.2051 best_f1=0.8046 best_th=0.1220
Epoch 20 | tr_loss=0.0442 val_loss=0.2006 best_f1=0.8046 best_th=0.1220
Epoch 25 | tr_loss=0.0191 val_loss=0.2321 best_f1=0.8046 best_th=0.1220
Epoch 30 | tr_loss=0.0126 val_loss=0.2584 best_f1=0.8095 best_th=0.1400
Fold 4 | P=0.9189 R=0.7234 F1=0.8095 (th=0.1400)

===== Fold 5/5 =====
Train pages: 26 Val pages: 7


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 05 | tr_loss=0.3820 val_loss=0.5232 best_f1=0.8986 best_th=0.1030
Epoch 10 | tr_loss=0.1538 val_loss=0.4255 best_f1=0.8986 best_th=0.1030
Epoch 15 | tr_loss=0.0535 val_loss=0.4527 best_f1=0.8986 best_th=0.1030
Epoch 20 | tr_loss=0.0283 val_loss=0.4995 best_f1=0.8986 best_th=0.1030
Epoch 25 | tr_loss=0.0217 val_loss=0.3667 best_f1=0.9147 best_th=0.1280
Epoch 30 | tr_loss=0.0199 val_loss=0.3727 best_f1=0.9231 best_th=0.0820
Fold 5 | P=0.9836 R=0.8696 F1=0.9231 (th=0.0820)

===== CV Summary =====
Mean F1: 0.8919
Mean threshold: 0.1238
Using threshold for final eval: 0.1238


In [12]:
# best_th_cv = 0.1238

In [13]:
# Final train on all data -----------------------------------------------------
print("\nTraining final model on all training data...")
final_train_df = df[df["source"].isin(cv_sources)].copy()
final_num_mean, final_num_std = compute_num_stats(final_train_df, NUM_COLS)

train_loader = make_loader(
    final_train_df, tokenizer,
    TAG_VOCAB, PARENT_TAG_VOCAB,
    NUM_COLS, BOOL_COLS,
    final_num_mean, final_num_std,
    batch_size=cfg["training"]["batch_size"],
    max_tokens=cfg["model"]["max_tokens"],
    shuffle=True
)

model, optimizer = init_model_and_optim(
    cfg,
    tag_vocab_size=len(TAG_VOCAB),
    parent_tag_vocab_size=len(PARENT_TAG_VOCAB),
    num_numeric_features=len(NUM_COLS),
    num_bool_features=len(BOOL_COLS),
    device=DEVICE
)

loss_fn = bio_loss(DEVICE)
# loss_fn = weighted_bio_loss(final_train_df, DEVICE)
# loss_fn = weighted_bio_loss(final_train_df, DEVICE, weight_cap=5.0)

set_bert_trainable(model, False)

for epoch in range(cfg["training"]["epochs"]):
    if epoch == cfg["training"]["freeze_epochs"]:
        set_bert_trainable(model, True)

    tr_loss = run_epoch(model, optimizer, train_loader, loss_fn, DEVICE, training=True)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:02d} | tr_loss={tr_loss:.4f}")

print("Final model training done.")


Training final model on all training data...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 05 | tr_loss=0.4804
Epoch 10 | tr_loss=0.3036
Epoch 15 | tr_loss=0.1391
Epoch 20 | tr_loss=0.0985
Epoch 25 | tr_loss=0.0489
Epoch 30 | tr_loss=0.0345
Final model training done.


In [14]:
test_loader = make_loader(
    test_df, tokenizer,
    TAG_VOCAB, PARENT_TAG_VOCAB,
    NUM_COLS, BOOL_COLS,
    final_num_mean, final_num_std,
    batch_size=cfg["training"]["batch_size"],
    max_tokens=cfg["model"]["max_tokens"],
    shuffle=False
)

p, r, f1 = boundary_metrics_peak(
    test_loader, model, DEVICE,
    threshold=best_th_cv,
    nms_k=cfg["inference"]["nms_k"],
    min_gap=cfg["inference"]["min_gap"],
    tol=cfg["inference"]["tol"]
)

print(f"\n===== HOLDOUT TEST =====")
print(f"START: P={p:.4f} R={r:.4f} F1={f1:.4f}")
print(f"Threshold used: {best_th_cv:.3f}")


===== HOLDOUT TEST =====
START: P=0.9167 R=1.0000 F1=0.9565
Threshold used: 0.124


In [15]:
# ── PER PAGE BOUNDARY VERIFICATION ────────────────────────────────────────
pages = collect_page_probs_and_truth(test_loader, model, DEVICE)

for i, (probs_full, prob_B, true_start) in enumerate(pages):
    source = test_sources[i]
    page_df = test_df[test_df["source"] == source].sort_values("rendering_order").reset_index(drop=True)

    true_starts = np.where(true_start == 1)[0].tolist()
    pred_starts = pick_starts_from_probs(
        prob_B,
        threshold=best_th_cv,   # use debug threshold
        nms_k=cfg["inference"]["nms_k"],
        min_gap=cfg["inference"]["min_gap"]
    )

    tp, fp, fn = start_prf_with_tolerance(true_starts, pred_starts, tol=cfg["inference"]["tol"])
    p, r, f1 = compute_prf(tp, fp, fn)

    print(f"\n========== Page: {source} ==========")
    print(f"Nodes: {len(page_df)}")
    print(f"True starts:  {true_starts}")
    print(f"Pred starts:  {pred_starts}")
    print(f"P={p:.4f} R={r:.4f} F1={f1:.4f}")

    print(f"\n--- Probability window around each true start ---")
    for idx in true_starts:
        lo = max(0, idx - 2)
        hi = min(len(prob_B), idx + 3)
        print(f"\nWindow around true start {idx}:")
        for j in range(lo, hi):
            print(f"  Index {j:3d} | Prob_B={prob_B[j]:.3f}")
            
    bio_labels = {0: "O", 1: "B", 2: "I"}
    print(f"\n--- Full node table ---")
    for j, row in page_df.iterrows():
        marker = ""
        if j in true_starts: marker += " [TRUE]"
        if j in pred_starts: marker += " [PRED]"
        txt = str(row.get("text_context", ""))[:50]
        pred_bio = bio_labels[probs_full[j].argmax()]
        print(f"  {j:3d} | BIO={pred_bio} | ProbB={prob_B[j]:.3f} | {txt}{marker}")


========== Page: neacac_spring.net_pattern_labeled ==========
Nodes: 233
True starts:  [148, 155, 161, 167, 173, 179, 185]
Pred starts:  [149, 156, 161, 167, 173, 179, 185, 189]
P=0.8750 R=1.0000 F1=0.9333

--- Probability window around each true start ---

Window around true start 148:
  Index 146 | Prob_B=0.001
  Index 147 | Prob_B=0.002
  Index 148 | Prob_B=0.011
  Index 149 | Prob_B=0.128
  Index 150 | Prob_B=0.003

Window around true start 155:
  Index 153 | Prob_B=0.001
  Index 154 | Prob_B=0.001
  Index 155 | Prob_B=0.808
  Index 156 | Prob_B=0.828
  Index 157 | Prob_B=0.004

Window around true start 161:
  Index 159 | Prob_B=0.002
  Index 160 | Prob_B=0.001
  Index 161 | Prob_B=0.493
  Index 162 | Prob_B=0.053
  Index 163 | Prob_B=0.004

Window around true start 167:
  Index 165 | Prob_B=0.002
  Index 166 | Prob_B=0.001
  Index 167 | Prob_B=0.836
  Index 168 | Prob_B=0.525
  Index 169 | Prob_B=0.002

Window around true start 173:
  Index 171 | Prob_B=0.002
  Index 172 | Prob_B

In [17]:
# Save checkpoint ----------------------------------------------------------------
checkpoint = {
    # model weights
    "model_state_dict": model.state_dict(),

    # vocabs needed to rebuild the model and dataset
    "label2id": label2id,
    "id2label": id2label,
    "tag_vocab": TAG_VOCAB,
    "parent_tag_vocab": PARENT_TAG_VOCAB,

    # normalization stats needed for inference
    "num_mean": final_num_mean,
    "num_std": final_num_std,

    # feature columns so inference knows what to expect
    "num_cols": NUM_COLS,
    "bool_cols": BOOL_COLS,

    # best threshold from CV
    "best_th": best_th_cv,

    # config used for this run
    "cfg": cfg,
}

torch.save(checkpoint, "/kaggle/working//dom_extractor_checkpoint_final.pt")
print("Checkpoint saved to /kaggle/working//dom_extractor_checkpoint.pt")

Checkpoint saved to /kaggle/working//dom_extractor_checkpoint.pt
